# 16S and metabolomics CTF Analyses of the Longitudinal Acne Study

Date last updated: 1/7/2026

Notebook author: Britta De Pessemier, Yang Chen 

Data analysis by: Tyler Myers, Britta De Pessemier, and Yang Chen

In [352]:
# Import essential Python packages
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Ellipse
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
import hashlib
from Bio import SeqIO

# Scientific and statistical packages
import scipy.stats as ss
from skbio import DistanceMatrix
from skbio.stats.distance import permanova

# BIOM and Qiime2 related packages
import biom
import qiime2 as q2
from biom import load_table
from qiime2 import Artifact
import h5py

# Gemelli packages
from qiime2 import Artifact, Metadata
from qiime2.plugins import gemelli
from skbio.stats.ordination import OrdinationResults

# Stats
from skbio.stats.distance import DistanceMatrix
from skbio.stats.distance import permanova
from scipy.spatial.distance import pdist, squareform
from statsmodels.stats.multitest import multipletests
import itertools
from itertools import combinations

# Other
import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message="You passed a edgecolor/edgecolors .*"
)


## Import table and metadata

In [353]:
biom_paths = {
    "16S_V1-V3": "../Data/16S/Tables/from_Qiita/179426_feature-table_16S_V1V3_rare-11054_sampleIDfixed.biom",
    "16S_V4": "../Data/16S/Tables/from_Qiita/174951_feature-table_16S_V4_rare-3769_sampleIDfixed.biom",
    "metabolomics": "../Data/metabolomics/Run3_10252024/output/metabolomics_table_final_sampleIDfixed.biom"
}

metadata_path = "../Metadata/metadata_final_22102024.tsv"
metadata_df = pd.read_csv(metadata_path, sep="\t")
metadata_df = metadata_df.set_index("SampleID")
metadata_df.index.name = '#SampleID'

In [354]:
# Make qiime2 compatible metadata
qiime_md = Metadata(metadata_df)

## Plotting preparation

In [355]:
def draw_ellipse(mean, cov, ax, color, scale_factor=1.5):
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
    width, height = 2 * np.sqrt(eigenvalues) * scale_factor

    ell = Ellipse(
        xy=mean,
        width=width,
        height=height,
        angle=angle,
        edgecolor=color,
        facecolor=color,
        alpha=0.2,
        linewidth=0.5
    )
    ax.add_patch(ell)

In [356]:
group_colors = {
    "Healthy": "#000096",
    "Acne_L": "#ff676c",
    "Acne_NL": "#17c6d5"
}

group_name_mapping = {
    "Healthy": "Healthy",
    "Acne_L": "Acne Lesional",
    "Acne_NL": "Acne Non-lesional"
}

In [357]:
def build_tensor_robust(table, md, individual_id, state_column):
    """
    Build gemelli CTF tensor across gemelli versions.
    """

    b = build()

    if hasattr(b, "__call__"):
        return b(table, md, individual_id, state_column)

    if hasattr(b, "fit_transform"):
        return b.fit_transform(
            table=table,
            sample_metadata=md,
            individual_id_column=individual_id,
            state_column=state_column
        )

    if hasattr(b, "build"):
        return b.build(
            table,
            md,
            individual_id_column=individual_id,
            state_column=state_column
        )

    raise RuntimeError("Unsupported gemelli build API")


In [358]:
def calculate_pairwise_permanova(ordination_df, group_column='Group', n_permutations=999):
    """
    Calculate pairwise PERMANOVA between groups
    """
    # Create distance matrix from ordination coordinates
    coords = ordination_df[['PC1', 'PC2', 'PC3']].values
    dist_array = squareform(pdist(coords, metric='euclidean'))
    
    # Create DistanceMatrix object with sample IDs
    dm = DistanceMatrix(dist_array, ids=ordination_df.index.tolist())
    
    # Get unique groups
    groups = ordination_df[group_column].unique()
    group_pairs = list(combinations(groups, 2))
    
    # Calculate pairwise PERMANOVA
    pairwise_results = {}
    
    for group1, group2 in group_pairs:
        # Get sample IDs for the two groups
        ids1 = ordination_df[ordination_df[group_column] == group1].index.tolist()
        ids2 = ordination_df[ordination_df[group_column] == group2].index.tolist()
        pair_ids = ids1 + ids2
        
        # Filter distance matrix for these samples
        pair_dm = dm.filter(pair_ids)
        
        # Create grouping vector
        pair_grouping = [group1] * len(ids1) + [group2] * len(ids2)
        
        # Run PERMANOVA
        result = permanova(pair_dm, pair_grouping, permutations=n_permutations)
        
        pairwise_results[f"{group1}_vs_{group2}"] = {
            'statistic': result['test statistic'],
            'p_value': result['p-value'],
            'R2': result['test statistic'] / (result['number of groups'] - 1)  # pseudo R²
        }
    
    return pairwise_results

## Run Compositional Tensor Factorization (CTF) (C1 - left cheek)

In [359]:
ctf_results = {}

for region, biom_path in biom_paths.items():

    print(f"\nRunning CTF for {region}")

    # Load BIOM
    table_artifact = Artifact.import_data(
        "FeatureTable[Frequency]",
        biom_path
    )
    table_df = table_artifact.view(pd.DataFrame)

    # Match samples between BIOM and metadata
    shared_samples = table_df.index.intersection(metadata_df.index)
    if shared_samples.empty:
        raise ValueError(f"No shared samples for {region}")

    table_df = table_df.loc[shared_samples]
    md_all = metadata_df.loc[shared_samples].copy()

    # REMOVE C2 SAMPLES HERE
    keep_idx = md_all["c_zone"] != "C2"

    table_df = table_df.loc[keep_idx]
    md_all = md_all.loc[keep_idx]

    print(f"  Samples after removing C2: {table_df.shape[0]}")

    if table_df.empty:
        raise ValueError(f"All samples removed after filtering C2 for {region}")

    # Re-import filtered table
    table_art = Artifact.import_data(
        "FeatureTable[Frequency]",
        table_df
    )

    # QIIME metadata
    md_all.index.name = "#SampleID"
    qiime_md = Metadata(md_all)

    # RUN CTF (C2 removed)
    ctf_res = gemelli.actions.ctf(
        table=table_art,
        sample_metadata=qiime_md,
        individual_id_column="subject_ID",
        state_column="day",
        n_components=3
    )

    # Subject biplot
    ordination = ctf_res.subject_biplot.view(OrdinationResults)

    spca_df = ordination.samples.iloc[:, :3].copy()
    spca_df.columns = ["PC1", "PC2", "PC3"]

    # Subject-level metadata
    subj_md = (
        md_all
        .drop_duplicates("subject_ID")
        .set_index("subject_ID")
    )

    spca_df["Group"] = spca_df.index.map(subj_md["group"])
    spca_df["day"] = spca_df.index.map(subj_md["day"])
    spca_df["c_zone"] = spca_df.index.map(subj_md["c_zone"])

    # Store results
    ctf_results[region] = {
        "ordination": ordination,
        "spca_df": spca_df,
        "metadata": subj_md
    }

    feature_df = (
    ordination.features
    .iloc[:, :3]
    .copy()
)

    # Extract feature loadings
    feature_df.columns = ["PC1", "PC2", "PC3"]
    feature_df["feature_id"] = feature_df.index

    # Sort by strongest overall loading (max abs across PCs)
    feature_df = (
        feature_df
        .assign(
            max_abs_loading=lambda x: x[["PC1", "PC2", "PC3"]].abs().max(axis=1)
        )
        .sort_values("max_abs_loading", ascending=False)
        .drop(columns="max_abs_loading")
    )

    # Save
    outdir = "../Analyses/16S/CTF/feature_loadings"
    os.makedirs(outdir, exist_ok=True)

    feature_df.to_csv(
        f"{outdir}/CTF_feature_loadings_{region}_left-cheek.tsv",
        sep="\t",
        index=False
    )

    # Save the subject biplot as qza
    qza_outdir = "../Analyses/16S/CTF/subject_biplot_qza"
    os.makedirs(qza_outdir, exist_ok=True)

    subject_biplot_qza = f"{qza_outdir}/CTF_subject_biplot_{region}_left-cheek.qza"

    ctf_res.subject_biplot.save(subject_biplot_qza)

    print(f"  Saved subject biplot QZA: {subject_biplot_qza}")



Running CTF for 16S_V1-V3
  Samples after removing C2: 128


/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/gemelli/preprocessing.py:968: RuntimeWarning: Subject(s) (PP_304,PP_319,PP_306,PP_317,PP_308,PP_310,PP_318,PP_305,PP_313,PP_314) contains multiple samples. Multiple subject counts will be meaned across samples by subject.
  warnings.warn(''.join(["Subject(s) (", str(duplicated_ids),


  Saved subject biplot QZA: ../Analyses/16S/CTF/subject_biplot_qza/CTF_subject_biplot_16S_V1-V3_left-cheek.qza

Running CTF for 16S_V4
  Samples after removing C2: 129


/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/gemelli/preprocessing.py:968: RuntimeWarning: Subject(s) (PP_304,PP_319,PP_306,PP_317,PP_308,PP_310,PP_318,PP_305,PP_313,PP_314) contains multiple samples. Multiple subject counts will be meaned across samples by subject.
  warnings.warn(''.join(["Subject(s) (", str(duplicated_ids),


  Saved subject biplot QZA: ../Analyses/16S/CTF/subject_biplot_qza/CTF_subject_biplot_16S_V4_left-cheek.qza

Running CTF for metabolomics
  Samples after removing C2: 158


/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/gemelli/preprocessing.py:968: RuntimeWarning: Subject(s) (PP_304,PP_319,PP_306,PP_317,PP_308,PP_310,PP_318,PP_305,PP_313,PP_314) contains multiple samples. Multiple subject counts will be meaned across samples by subject.
  warnings.warn(''.join(["Subject(s) (", str(duplicated_ids),


  Saved subject biplot QZA: ../Analyses/16S/CTF/subject_biplot_qza/CTF_subject_biplot_metabolomics_left-cheek.qza


In [360]:
np.random.seed(69)

# Manual layout presets per modality
LAYOUT_PRESETS = {
    "16S_V1-V3": {
        "group_legend_xy": (0.643, 0.0006),
        "permanova_xy": (0.632, 0.98),
        "legend_loc": "lower left"
    },
    "16S_V4": {
        "group_legend_xy": (0.643, 0.0006),
        "permanova_xy": (0.643, 0.98),
        "legend_loc": "lower left"
    },
    "metabolomics": {
        "group_legend_xy": (0.643, 0.0006),
        "permanova_xy": (0.02, 0.98),
        "legend_loc": "lower left"
    }
}

# Group name abbreviations for legend and annotations
group_abbrev = {
    'Healthy': 'H',
    'Acne_L': 'AL',
    'Acne_NL': 'ANL'
}

for region, res in ctf_results.items():

    ordination = res["ordination"]
    spca_df = res["spca_df"]

    # Select layout preset for this modality
    if region not in LAYOUT_PRESETS:
        raise ValueError(f"No layout preset defined for region: {region}")
    layout = LAYOUT_PRESETS[region]

    # Run pairwise PERMANOVA
    permanova_results = calculate_pairwise_permanova(
        spca_df,
        group_column="Group"
    )

    # FDR correction
    comparisons = list(permanova_results.keys())
    p_values = [v["p_value"] for v in permanova_results.values()]
    _, qvals, _, _ = multipletests(p_values, method="fdr_bh")

    for i, comp in enumerate(comparisons):
        permanova_results[comp]["adjusted_p_value"] = qvals[i]

    # Axis variance explained
    pc1_var = ordination.proportion_explained[0] * 100
    pc2_var = ordination.proportion_explained[1] * 100

    fig, ax = plt.subplots(figsize=(8, 8))

    # Scatter points and ellipses
    plot_df = spca_df.copy()

    for group, df_g in plot_df.groupby("Group"):

        color = group_colors[group]
        n_subjects = df_g.shape[0]
        label = f"{group_abbrev[group]} (n={n_subjects} subj. over time)"

        ax.scatter(
            df_g["PC1"],
            df_g["PC2"],
            s=120,
            color=color,
            edgecolor="black",
            linewidth=0.6,
            alpha=1,
            label=label
        )

        if len(df_g) > 2:
            pts = df_g[["PC1", "PC2"]].values
            draw_ellipse(
                mean=pts.mean(axis=0),
                cov=np.cov(pts, rowvar=False),
                ax=ax,
                color=color
            )

    # Axis labels and title
    ax.set_xlabel(f"PC1 ({pc1_var:.2f}%)", fontsize=16)
    ax.set_ylabel(f"PC2 ({pc2_var:.2f}%)", fontsize=16)
    title_region = region.replace("16S_", "16S ")
    ax.set_title(f"CTF {title_region} (left cheek)", fontsize=18)

    # Group legend with manual position per modality
    legend = ax.legend(
        frameon=True,
        fancybox=True,
        bbox_to_anchor=layout["group_legend_xy"],
        loc=layout["legend_loc"]
    )

    # Explicit legend frame styling
    frame = legend.get_frame()
    frame.set_facecolor("white")
    frame.set_edgecolor("grey")
    frame.set_linewidth(1.0)
    frame.set_alpha(0.9)

    legend.get_frame().set_linewidth(1.0)

    # Build PERMANOVA annotation text
    annotation_lines = []
    for comp, stats in permanova_results.items():

        g1, g2 = comp.split("_vs_")
        g1 = group_abbrev.get(g1, g1)
        g2 = group_abbrev.get(g2, g2)

        q = stats["adjusted_p_value"]

        if q < 0.001:
            ptxt = f"*** p={q:.3f}"
        elif q < 0.01:
            ptxt = f"** p={q:.3f}"
        elif q < 0.05:
            ptxt = f"* p={q:.3f}"
        else:
            ptxt = f"p={q:.3f}"

        annotation_lines.append(
            f"{g1} vs {g2}: {ptxt}, F={stats['statistic']:.3f}"
        )

    full_annotation = (
        "PERMANOVA (FDR-adjusted):\n" +
        "\n".join(annotation_lines)
    )

    # PERMANOVA annotation box with manual position per modality
    ax.text(
        layout["permanova_xy"][0],
        layout["permanova_xy"][1],
        full_annotation,
        transform=ax.transAxes,
        fontsize=11,
        verticalalignment="top",
        horizontalalignment="left",
        bbox=dict(
            boxstyle="round,pad=0.5",
            facecolor="white",
            edgecolor="grey",
            linewidth=1,
            alpha=0.75
        )
    )

    # Save figure
    plt.tight_layout()
    plt.savefig(
        f"../Figures/Main/Figure_2/CTF_{region}_left-cheek.png",
        dpi=600,
        bbox_inches="tight"
    )
    plt.close()

    # Print PERMANOVA results to console
    print(f"\n=== PERMANOVA results for {region} ===")
    for comp, stats in permanova_results.items():
        print(f"{comp}:")
        print(f"  Raw p-value: {stats['p_value']:.3f}")
        print(f"  FDR-adjusted q-value: {stats['adjusted_p_value']:.3f}")
        print(f"  F-statistic: {stats['statistic']:.3f}")



=== PERMANOVA results for 16S_V1-V3 ===
Healthy_vs_Acne_NL:
  Raw p-value: 0.071
  FDR-adjusted q-value: 0.106
  F-statistic: 2.333
Healthy_vs_Acne_L:
  Raw p-value: 0.005
  FDR-adjusted q-value: 0.015
  F-statistic: 4.491
Acne_NL_vs_Acne_L:
  Raw p-value: 0.592
  FDR-adjusted q-value: 0.592
  F-statistic: 0.725

=== PERMANOVA results for 16S_V4 ===
Healthy_vs_Acne_NL:
  Raw p-value: 0.024
  FDR-adjusted q-value: 0.072
  F-statistic: 3.435
Healthy_vs_Acne_L:
  Raw p-value: 0.070
  FDR-adjusted q-value: 0.105
  F-statistic: 2.373
Acne_NL_vs_Acne_L:
  Raw p-value: 0.488
  FDR-adjusted q-value: 0.488
  F-statistic: 0.756

=== PERMANOVA results for metabolomics ===
Healthy_vs_Acne_NL:
  Raw p-value: 0.273
  FDR-adjusted q-value: 0.401
  F-statistic: 1.367
Healthy_vs_Acne_L:
  Raw p-value: 0.001
  FDR-adjusted q-value: 0.003
  F-statistic: 6.282
Acne_NL_vs_Acne_L:
  Raw p-value: 0.401
  FDR-adjusted q-value: 0.401
  F-statistic: 1.147


## Run Compositional Tensor Factorization (CTF) (C2 - right cheek)

In [361]:
ctf_results = {}

for region, biom_path in biom_paths.items():

    print(f"\nRunning CTF for {region}")

    # Load BIOM
    table_artifact = Artifact.import_data(
        "FeatureTable[Frequency]",
        biom_path
    )
    table_df = table_artifact.view(pd.DataFrame)

    # Match samples between BIOM and metadata
    shared_samples = table_df.index.intersection(metadata_df.index)
    if shared_samples.empty:
        raise ValueError(f"No shared samples for {region}")

    table_df = table_df.loc[shared_samples]
    md_all = metadata_df.loc[shared_samples].copy()

    # REMOVE C1 SAMPLES HERE
    keep_idx = md_all["c_zone"] != "C1"

    table_df = table_df.loc[keep_idx]
    md_all = md_all.loc[keep_idx]

    print(f"  Samples after removing C1: {table_df.shape[0]}")

    if table_df.empty:
        raise ValueError(f"All samples removed after filtering C2 for {region}")

    # Re-import filtered table
    table_art = Artifact.import_data(
        "FeatureTable[Frequency]",
        table_df
    )

    # QIIME metadata
    md_all.index.name = "#SampleID"
    qiime_md = Metadata(md_all)

    # RUN CTF (C1 removed)
    ctf_res = gemelli.actions.ctf(
        table=table_art,
        sample_metadata=qiime_md,
        individual_id_column="subject_ID",
        state_column="day",
        n_components=3
    )

    # Subject biplot
    ordination = ctf_res.subject_biplot.view(OrdinationResults)

    spca_df = ordination.samples.iloc[:, :3].copy()
    spca_df.columns = ["PC1", "PC2", "PC3"]

    # Subject-level metadata
    subj_md = (
        md_all
        .drop_duplicates("subject_ID")
        .set_index("subject_ID")
    )

    spca_df["Group"] = spca_df.index.map(subj_md["group"])
    spca_df["day"] = spca_df.index.map(subj_md["day"])
    spca_df["c_zone"] = spca_df.index.map(subj_md["c_zone"])

    # Store results
    ctf_results[region] = {
        "ordination": ordination,
        "spca_df": spca_df,
        "metadata": subj_md
    }

    # Extract feature loadings
    feature_df = (
        ordination.features
        .iloc[:, :3]
        .copy()
    )
    feature_df.columns = ["PC1", "PC2", "PC3"]
    feature_df["feature_id"] = feature_df.index

    # Sort by strongest overall loading (max abs across PCs)
    feature_df = (
        feature_df
        .assign(
            max_abs_loading=lambda x: x[["PC1", "PC2", "PC3"]].abs().max(axis=1)
        )
        .sort_values("max_abs_loading", ascending=False)
        .drop(columns="max_abs_loading")
    )

    # Save
    outdir = "../Analyses/16S/CTF/feature_loadings"
    os.makedirs(outdir, exist_ok=True)

    feature_df.to_csv(
        f"{outdir}/CTF_feature_loadings_{region}_right-cheek.tsv",
        sep="\t",
        index=False
    )

    # Save the subject biplot as qza
    qza_outdir = "../Analyses/16S/CTF/subject_biplot_qza"
    os.makedirs(qza_outdir, exist_ok=True)

    subject_biplot_qza = f"{qza_outdir}/CTF_subject_biplot_{region}_right-cheek.qza"

    ctf_res.subject_biplot.save(subject_biplot_qza)

    print(f"  Saved subject biplot QZA: {subject_biplot_qza}")


Running CTF for 16S_V1-V3
  Samples after removing C1: 128


/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/gemelli/preprocessing.py:968: RuntimeWarning: Subject(s) (PP_304,PP_319,PP_306,PP_317,PP_308,PP_310,PP_318,PP_305,PP_313,PP_314) contains multiple samples. Multiple subject counts will be meaned across samples by subject.
  warnings.warn(''.join(["Subject(s) (", str(duplicated_ids),


  Saved subject biplot QZA: ../Analyses/16S/CTF/subject_biplot_qza/CTF_subject_biplot_16S_V1-V3_right-cheek.qza

Running CTF for 16S_V4
  Samples after removing C1: 133


/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/gemelli/preprocessing.py:968: RuntimeWarning: Subject(s) (PP_304,PP_319,PP_306,PP_317,PP_308,PP_310,PP_318,PP_305,PP_313,PP_314) contains multiple samples. Multiple subject counts will be meaned across samples by subject.
  warnings.warn(''.join(["Subject(s) (", str(duplicated_ids),


  Saved subject biplot QZA: ../Analyses/16S/CTF/subject_biplot_qza/CTF_subject_biplot_16S_V4_right-cheek.qza

Running CTF for metabolomics
  Samples after removing C1: 158


/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/gemelli/preprocessing.py:968: RuntimeWarning: Subject(s) (PP_304,PP_319,PP_306,PP_317,PP_308,PP_310,PP_318,PP_305,PP_313,PP_314) contains multiple samples. Multiple subject counts will be meaned across samples by subject.
  warnings.warn(''.join(["Subject(s) (", str(duplicated_ids),


  Saved subject biplot QZA: ../Analyses/16S/CTF/subject_biplot_qza/CTF_subject_biplot_metabolomics_right-cheek.qza


In [362]:
np.random.seed(69)

# Layout presets per modality (reuse same dict as left cheek)
LAYOUT_PRESETS = {
    "16S_V1-V3": {
        "group_legend_xy": (0.643, 0.0006),
        "permanova_xy": (0.632, 0.98),
        "legend_loc": "lower left"
    },
    "16S_V4": {
        "group_legend_xy": (0.643, 0.0006),
        "permanova_xy": (0.643, 0.98),
        "legend_loc": "lower left"
    },
    "metabolomics": {
        "group_legend_xy": (0.643, 0.0006),
        "permanova_xy": (0.02, 0.98),
        "legend_loc": "lower left"
    }
}

group_abbrev = {
    'Healthy': 'H',
    'Acne_L': 'AL',
    'Acne_NL': 'ANL'
}

for region, res in ctf_results.items():

    ordination = res["ordination"]
    spca_df = res["spca_df"]

    if region not in LAYOUT_PRESETS:
        raise ValueError(f"No layout preset defined for region: {region}")
    layout = LAYOUT_PRESETS[region]

    # PERMANOVA
    permanova_results = calculate_pairwise_permanova(
        spca_df, group_column="Group"
    )

    comparisons = list(permanova_results.keys())
    p_values = [v["p_value"] for v in permanova_results.values()]
    _, qvals, _, _ = multipletests(p_values, method="fdr_bh")

    for i, comp in enumerate(comparisons):
        permanova_results[comp]["adjusted_p_value"] = qvals[i]

    pc1_var = ordination.proportion_explained[0] * 100
    pc2_var = ordination.proportion_explained[1] * 100

    fig, ax = plt.subplots(figsize=(8, 8))

    # Scatter + ellipses
    for group, df_g in spca_df.groupby("Group"):

        color = group_colors[group]
        n_samples = df_g.shape[0]
        label = f"{group_abbrev[group]} (n={n_samples} subj. over time)"

        ax.scatter(
            df_g["PC1"],
            df_g["PC2"],
            s=120,
            color=color,
            edgecolor="black",
            linewidth=0.6,
            alpha=1,
            label=label
        )

        if len(df_g) > 2:
            pts = df_g[["PC1", "PC2"]].values
            draw_ellipse(
                mean=pts.mean(axis=0),
                cov=np.cov(pts, rowvar=False),
                ax=ax,
                color=color
            )

    # Labels & title
    ax.set_xlabel(f"PC1 ({pc1_var:.2f}%)", fontsize=16)
    ax.set_ylabel(f"PC2 ({pc2_var:.2f}%)", fontsize=16)

    title_region = region.replace("16S_", "16S ")
    ax.set_title(f"CTF {title_region} (right cheek)", fontsize=18)

    # Legend (layout-controlled)
    legend = ax.legend(
        frameon=True,
        fancybox=True,
        bbox_to_anchor=layout["group_legend_xy"],
        loc=layout["legend_loc"]
    )

    frame = legend.get_frame()
    frame.set_facecolor("white")
    frame.set_edgecolor("grey")
    frame.set_linewidth(1.0)
    frame.set_alpha(0.9)

    # PERMANOVA annotation
    annotation_lines = []
    for comp, stats in permanova_results.items():

        g1, g2 = comp.split("_vs_")
        g1 = group_abbrev.get(g1, g1)
        g2 = group_abbrev.get(g2, g2)

        q = stats["adjusted_p_value"]

        if q < 0.001:
            ptxt = f"*** p={q:.3f}"
        elif q < 0.01:
            ptxt = f"** p={q:.3f}"
        elif q < 0.05:
            ptxt = f"* p={q:.3f}"
        else:
            ptxt = f"p={q:.3f}"

        annotation_lines.append(
            f"{g1} vs {g2}: {ptxt}, F={stats['statistic']:.3f}"
        )

    full_annotation = (
        "PERMANOVA (FDR-adjusted):\n" +
        "\n".join(annotation_lines)
    )

    ax.text(
        layout["permanova_xy"][0],
        layout["permanova_xy"][1],
        full_annotation,
        transform=ax.transAxes,
        fontsize=11,
        verticalalignment="top",
        horizontalalignment="left",
        bbox=dict(
            boxstyle="round,pad=0.5",
            facecolor="white",
            edgecolor="grey",
            linewidth=1,
            alpha=0.75
        )
    )

    # Save
    plt.tight_layout()
    plt.savefig(
        f"../Figures/Main/Figure_2/CTF_{region}_right-cheek.png",
        dpi=600,
        bbox_inches="tight"
    )
    plt.close()

    # Console output
    print(f"\n=== PERMANOVA results for {region} (right cheek) ===")
    for comp, stats in permanova_results.items():
        print(f"{comp}:")
        print(f"  Raw p-value: {stats['p_value']:.3f}")
        print(f"  FDR-adjusted q-value: {stats['adjusted_p_value']:.3f}")
        print(f"  F-statistic: {stats['statistic']:.3f}")



=== PERMANOVA results for 16S_V1-V3 (right cheek) ===
Healthy_vs_Acne_NL:
  Raw p-value: 0.001
  FDR-adjusted q-value: 0.003
  F-statistic: 4.316
Healthy_vs_Acne_L:
  Raw p-value: 0.028
  FDR-adjusted q-value: 0.042
  F-statistic: 2.868
Acne_NL_vs_Acne_L:
  Raw p-value: 0.120
  FDR-adjusted q-value: 0.120
  F-statistic: 2.238

=== PERMANOVA results for 16S_V4 (right cheek) ===
Healthy_vs_Acne_L:
  Raw p-value: 0.006
  FDR-adjusted q-value: 0.018
  F-statistic: 3.964
Healthy_vs_Acne_NL:
  Raw p-value: 0.019
  FDR-adjusted q-value: 0.029
  F-statistic: 3.215
Acne_L_vs_Acne_NL:
  Raw p-value: 0.479
  FDR-adjusted q-value: 0.479
  F-statistic: 0.933

=== PERMANOVA results for metabolomics (right cheek) ===
Healthy_vs_Acne_NL:
  Raw p-value: 0.136
  FDR-adjusted q-value: 0.204
  F-statistic: 2.043
Healthy_vs_Acne_L:
  Raw p-value: 0.001
  FDR-adjusted q-value: 0.003
  F-statistic: 8.927
Acne_NL_vs_Acne_L:
  Raw p-value: 0.449
  FDR-adjusted q-value: 0.449
  F-statistic: 1.055


## Map taxonomy of each ASV feature loading within each CTF model

In [363]:
ctf_dir = "../Analyses/16S/CTF/feature_loadings"

primer_sets = ["V1-V3", "V4"]
cheek_sides = ["left", "right"]

for primer_set in primer_sets:
    for cheek_side in cheek_sides:

        ctf_path = (
            f"{ctf_dir}/CTF_feature_loadings_16S_"
            f"{primer_set}_{cheek_side}-cheek.tsv"
        )

        if not os.path.exists(ctf_path):
            print(f"Skipping missing file: {ctf_path}")
            continue

        print(f"Annotating {os.path.basename(ctf_path)}")

        df = pd.read_csv(ctf_path, sep="\t")

        if "feature_id" not in df.columns:
            raise ValueError(f"'feature_id' column missing in {ctf_path}")

        # Select FASTA based on primer set
        if primer_set == "V1-V3":
            fasta_file = "../Data/16S/Fasta/179426_V1V3_ASVs.fasta"
        elif primer_set == "V4":
            fasta_file = "../Data/16S/Fasta/174951_V4_ASVs.fasta"
        else:
            raise ValueError(f"Unknown primer set: {primer_set}")

        # Build sequence → ASV-hash mapping from FASTA
        seq_to_hash = {}

        for record in SeqIO.parse(fasta_file, "fasta"):
            seq = str(record.seq).upper()
            asv_hash = record.id
            seq_to_hash[seq] = asv_hash

        # Map feature_id (sequence) → asv_hash
        df["asv_hash"] = df["feature_id"].str.upper().map(seq_to_hash)

        # Sanity check
        n_missing = df["asv_hash"].isna().sum()
        if n_missing > 0:
            raise ValueError(
                f"{n_missing} feature_id sequences in {ctf_path} "
                f"were not found in {os.path.basename(fasta_file)}"
            )

        # Write back to disk
        df.to_csv(ctf_path, sep="\t", index=False)


Annotating CTF_feature_loadings_16S_V1-V3_left-cheek.tsv
Annotating CTF_feature_loadings_16S_V1-V3_right-cheek.tsv
Annotating CTF_feature_loadings_16S_V4_left-cheek.tsv
Annotating CTF_feature_loadings_16S_V4_right-cheek.tsv


In [364]:
# Paths and configuration
ctf_dir = "../Analyses/16S/CTF/feature_loadings"

primer_sets = ["V1-V3", "V4"]
cheek_sides = ["left", "right"]

taxonomy_paths = {
    "V1-V3": "../Analyses/16S/SEPP/179426_V1V3_GG2_taxonomy/taxonomy.tsv",
    "V4": "../Analyses/16S/SEPP/174951_V4_GG2_taxonomy/taxonomy.tsv",
}

def extract_lowest_taxon(taxon):
    if pd.isna(taxon):
        return "Unclassified"

    parts = [p.strip() for p in taxon.split(";") if p.strip()]

    # Preferred rank order (lowest → highest)
    rank_prefixes = [
        "g__",  # genus
        "f__",  # family
        "o__",  # order
        "c__",  # class
        "p__",  # phylum
        "d__",  # domain
    ]

    for prefix in rank_prefixes:
        for p in parts:
            if p.startswith(prefix) and p != prefix:
                return p

    return "Unclassified"


# Load taxonomy mappings (asv_hash → Taxon)
tax_maps = {}

for primer_set, tax_path in taxonomy_paths.items():
    if not os.path.exists(tax_path):
        raise FileNotFoundError(f"Missing taxonomy file: {tax_path}")

    tax_df = pd.read_csv(tax_path, sep="\t")

    required_cols = {"Feature ID", "Taxon"}
    if not required_cols.issubset(tax_df.columns):
        raise ValueError(f"taxonomy.tsv malformed: {tax_path}")

    tax_maps[primer_set] = dict(
        zip(tax_df["Feature ID"], tax_df["Taxon"])
    )

# Annotate CTF feature loadings
for primer_set in primer_sets:
    for cheek_side in cheek_sides:

        ctf_path = (
            f"{ctf_dir}/CTF_feature_loadings_16S_"
            f"{primer_set}_{cheek_side}-cheek.tsv"
        )

        if not os.path.exists(ctf_path):
            print(f"Skipping missing file: {ctf_path}")
            continue

        print(f"Annotating taxonomy: {os.path.basename(ctf_path)}")

        df = pd.read_csv(ctf_path, sep="\t")

        if "asv_hash" not in df.columns:
            raise ValueError(f"'asv_hash' column missing in {ctf_path}")

        # Map full taxonomy
        df["Taxon"] = df["asv_hash"].map(tax_maps[primer_set])

        # Sanity check
        missing = df["Taxon"].isna().sum()
        if missing > 0:
            raise ValueError(
                f"{missing} ASVs in {ctf_path} "
                f"were not found in {primer_set} taxonomy.tsv"
            )

        # Add lowest resolved taxon
        df["lowest_taxon"] = df["Taxon"].apply(extract_lowest_taxon)

        # Write annotated file
        df.to_csv(ctf_path, sep="\t", index=False)


Annotating taxonomy: CTF_feature_loadings_16S_V1-V3_left-cheek.tsv
Annotating taxonomy: CTF_feature_loadings_16S_V1-V3_right-cheek.tsv
Annotating taxonomy: CTF_feature_loadings_16S_V4_left-cheek.tsv
Annotating taxonomy: CTF_feature_loadings_16S_V4_right-cheek.tsv


## Plot feature importances

In [365]:
# Paths
CTF_DIR = "../Analyses/16S/CTF/feature_loadings"
FIG_OUT_DIR = "../Figures/Supplementary/Suppl_Figure_2/"
os.makedirs(FIG_OUT_DIR, exist_ok=True)

# Parameters
TOP_N = 10
PRIMERS = ["V1-V3", "V4"]
CHEEKS = ["left", "right"]

# Per-cheek CTF feature importance
for primer in PRIMERS:
    for cheek in CHEEKS:

        tsv_path = f"{CTF_DIR}/CTF_feature_loadings_16S_{primer}_{cheek}-cheek.tsv"

        if not os.path.exists(tsv_path):
            print(f"Skipping missing file: {tsv_path}")
            continue

        print(f"\nProcessing 16S {primer} {cheek} cheek")

        df = pd.read_csv(tsv_path, sep="\t")

        required_cols = {"feature_id", "PC1", "PC2", "PC3", "asv_hash", "Taxon", "lowest_taxon"}
        if not required_cols.issubset(df.columns):
            raise ValueError(f"Missing required columns in {tsv_path}")

        # CTF loading magnitude
        df["importance"] = np.sqrt(
            df["PC1"]**2 + df["PC2"]**2 + df["PC3"]**2
        )

        # Rank features
        df = df.sort_values("importance", ascending=False)

        # Overwrite TSV with importance + labels
        df.to_csv(tsv_path, sep="\t", index=False)

        # Plot top N features
        plot_df = df.head(TOP_N).iloc[::-1]
        y_pos = np.arange(len(plot_df))

        fig, ax = plt.subplots(figsize=(6, 0.35 * TOP_N + 1))
        ax.barh(y_pos, plot_df["importance"])
        ax.set_yticks(y_pos)
        ax.set_yticklabels(plot_df["lowest_taxon"] + " ASV")

        ax.set_xlabel("CTF loading magnitude (√(PC1² + PC2² + PC3²))")
        ax.set_title(f"Top {TOP_N} CTF features\n16S {primer} {cheek} cheek")

        plt.tight_layout()

        out_png = os.path.join(
            FIG_OUT_DIR,
            f"16S_{primer}_{cheek}-cheek_top{TOP_N}_CTF_features.png"
        )
        plt.savefig(out_png, dpi=600)
        plt.close()

        print(f"Saved {out_png}")

print("\nAll per-cheek CTF feature importance plots generated.")



Processing 16S V1-V3 left cheek
Saved ../Figures/Supplementary/Suppl_Figure_2/16S_V1-V3_left-cheek_top10_CTF_features.png

Processing 16S V1-V3 right cheek
Saved ../Figures/Supplementary/Suppl_Figure_2/16S_V1-V3_right-cheek_top10_CTF_features.png

Processing 16S V4 left cheek
Saved ../Figures/Supplementary/Suppl_Figure_2/16S_V4_left-cheek_top10_CTF_features.png

Processing 16S V4 right cheek
Saved ../Figures/Supplementary/Suppl_Figure_2/16S_V4_right-cheek_top10_CTF_features.png

All per-cheek CTF feature importance plots generated.
